# Explore RSNA Pneumonia

This notebook uses the standard PyHealth flow: RSNA dataset → `set_task()` → `SampleDataset` → `get_dataloader()`. `dev=True` limits exploration to 1000 patients. RSNA has no segmentation mask; `overlay_mask` is accepted for helper consistency. RSNA requires a manual Kaggle download; data and caches default to `~/pyhealth-imaging-data` and `~/.cache/pyhealth-imaging` (`PYHEALTH_IMAGING_DATA` / `PYHEALTH_IMAGING_CACHE` override them).

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from pyhealth.datasets import RSNAPneumoniaDataset, get_dataloader

# Override these with environment variables if your data/cache live elsewhere:
#   PYHEALTH_IMAGING_DATA
#   PYHEALTH_IMAGING_CACHE
DATA_ROOT = Path(
    os.environ.get("PYHEALTH_IMAGING_DATA", "~/pyhealth-imaging-data")
).expanduser()
CACHE_DIR = Path(
    os.environ.get("PYHEALTH_IMAGING_CACHE", "~/.cache/pyhealth-imaging")
).expanduser()
CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("TORCH_HOME", str(CACHE_DIR / "torch"))
print("Data directory:", DATA_ROOT)
print("Cache directory:", CACHE_DIR)

# Processed caches and generated metadata are stored here, outside the repo.
METADATA_DIR = CACHE_DIR / "metadata"


dataset = RSNAPneumoniaDataset(
    root=DATA_ROOT / "RSNAPneumoniaDetection2018",
    cache_dir=CACHE_DIR,
    metadata_dir=str(METADATA_DIR / "rsna_pneumonia"),
    dev=True,
)
dataset.stats()


def show_rsna_samples(samples, indices, overlay_mask=False):
    fig, axes = plt.subplots(
        1, len(indices), figsize=(4 * len(indices), 4), squeeze=False
    )
    for col, index in enumerate(indices):
        sample = samples[index]
        image_tensor = samples.input_processors["image"].materialize(sample["image"])
        image = image_tensor[0].cpu()
        axes[0, col].imshow(image, cmap="gray")
        if overlay_mask and "mask" in sample:
            mask_tensor = samples.output_processors["mask"].materialize(sample["mask"])
            mask = mask_tensor[0].cpu()
            axes[0, col].imshow(
                np.ma.masked_where(mask == 0, mask), cmap="autumn", alpha=0.35
            )
        axes[0, col].set_title(
            f"{sample['patient_id'][:8]}... label={int(sample['pneumonia'])}"
        )
        axes[0, col].axis("off")
    plt.show()


In [ ]:
patient_ids = dataset.unique_patient_ids
print(patient_ids[:5])
dataset.get_patient(patient_ids[0]).get_events()[0].attr_dict


In [ ]:
task = dataset.default_task
print(task.task_name)
print(task.input_schema)
print(task.output_schema)


In [ ]:
samples = dataset.set_task(task, num_workers=1)
print("samples:", len(samples))
print("input schema:", samples.input_schema)
print("output schema:", samples.output_schema)


In [ ]:
sample = samples[0]
image = samples.input_processors["image"].materialize(sample["image"])
print(sample.keys())
print("image:", image.shape, image.dtype)
print("pneumonia:", sample["pneumonia"])


In [ ]:
loader = get_dataloader(samples, batch_size=2)
batch = next(iter(loader))
print("batch image:", batch["image"].shape)
print("batch pneumonia:", batch["pneumonia"].shape)


In [ ]:
labels = [int(samples[i]["pneumonia"]) for i in range(len(samples))]
positive = labels.index(1)
negative = labels.index(0)
indices = [positive, negative]
for index, label in enumerate(labels):
    if len(indices) >= 3:
        break
    if index not in indices and label != labels[indices[0]]:
        indices.append(index)

show_rsna_samples(samples, indices, overlay_mask=False)


## Classification training

RSNA provides classification labels and bounding boxes, but no pixel-level segmentation masks. This section trains the existing `RSNAPneumoniaClassification` task with a torchvision ResNet18. Detection (bbox + mAP) is a separate task and is not covered by this notebook.

**Metric note:** the adjudicated positive rate is about 24%, so an all-negative classifier already gets ~76% accuracy. Accuracy is therefore not a good progress metric here; training reports ROC AUC and PR AUC as well.

**Overfitting control:** instead of fine-tuning all 11M parameters, the early residual stages stay frozen and only `layer4` plus the final `fc` are trained. The learning rate is lowered to `1e-5`, weight decay is raised to `1e-2`, and early stopping monitors validation loss (`patience=2`). This keeps the train/validation loss gap small while still reaching a useful ROC AUC. No data augmentation is added at this stage.

The final `evaluate()` loss can be lower than the last printed validation loss because `Trainer` restores the best checkpoint selected by validation loss, not the last epoch.


In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score

from pyhealth.datasets import (
    PatientSplit,
    RSNAPneumoniaDataset,
    get_dataloader,
)
from pyhealth.models import TorchvisionModel
from pyhealth.trainer import Trainer

torch.backends.cudnn.benchmark = True

EPOCHS = 10
PATIENCE = 2
BATCH_SIZE = 32
NUM_WORKERS = 8
PIN_MEMORY = torch.cuda.is_available()
LR = 1e-5
WEIGHT_DECAY = 1e-2
MAX_TRAIN_CASES = 8000  # set None to use the full training split
MAX_VAL_CASES = 2000
ANNOTATIONS_FILE = (
    DATA_ROOT
    / "RSNAPneumoniaDetection2018"
    / "pneumonia-challenge-annotations-adjudicated-kaggle_2018.json"
)

device = "cuda" if torch.cuda.is_available() else "cpu"

# Use the full 30k mapping for training; the exploration `dataset` above uses
# dev=True only to keep the first cells fast.
train_dataset = RSNAPneumoniaDataset(
    root=DATA_ROOT / "RSNAPneumoniaDetection2018",
    cache_dir=CACHE_DIR,
    metadata_dir=str(METADATA_DIR / "rsna_pneumonia"),
    dev=False,
    annotations_file=str(ANNOTATIONS_FILE) if ANNOTATIONS_FILE.exists() else None,
    label_source="auto",
)
print("label source:", train_dataset.label_source)
train_ds, val_ds = train_dataset.set_task(
    train_dataset.default_task,
    split=PatientSplit(ratios=(0.8, 0.2), seed=42),
    num_workers=1,
)
if MAX_TRAIN_CASES is not None and MAX_TRAIN_CASES < len(train_ds):
    rng = np.random.default_rng(42)
    train_ds = train_ds.subset(
        rng.choice(len(train_ds), MAX_TRAIN_CASES, replace=False)
    )
if MAX_VAL_CASES is not None and MAX_VAL_CASES < len(val_ds):
    rng = np.random.default_rng(43)
    val_ds = val_ds.subset(
        rng.choice(len(val_ds), MAX_VAL_CASES, replace=False)
    )

loader_kwargs = dict(num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
if NUM_WORKERS > 0:
    loader_kwargs.update(persistent_workers=True, prefetch_factor=2)
train_loader = get_dataloader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, **loader_kwargs
)
val_loader = get_dataloader(
    val_ds, batch_size=BATCH_SIZE, **loader_kwargs
)

try:
    model = TorchvisionModel(
        train_ds,
        model_name="resnet18",
        model_config={"weights": "DEFAULT"},
    )
except Exception as exc:
    print("Pretrained weights unavailable, using random init:", exc)
    model = TorchvisionModel(
        train_ds,
        model_name="resnet18",
        model_config={"weights": None},
    )

# Keep the ImageNet features in conv1-layer3; adapt only the last residual
# stage and the classifier. This is the main regularizer against overfitting.
for name, parameter in model.model.named_parameters():
    parameter.requires_grad = name.startswith("layer4") or name.startswith("fc")
print(
    "trainable parameters (M):",
    sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6,
)

positive_rate = pd.read_csv(train_dataset.metadata_path)["label"].mean()
print("positive rate:", positive_rate)

trainer = Trainer(
    model,
    device=device,
    metrics=["roc_auc", "pr_auc", "accuracy"],
)
trainer.train(
    train_loader,
    val_dataloader=val_loader,
    epochs=EPOCHS,
    optimizer_params={"lr": LR},
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=1.0,
    monitor="loss",
    monitor_criterion="min",
    patience=PATIENCE,
)

print("validation:", trainer.evaluate(val_loader))
y_true, y_prob, _ = trainer.inference(val_loader)
if len(np.unique(y_true)) > 1:
    print("roc_auc:", roc_auc_score(y_true.ravel(), y_prob.ravel()))

run_dir = CACHE_DIR / "runs"
run_dir.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), run_dir / "rsna_resnet18.ckpt")